<a href="https://colab.research.google.com/github/VarunPalrecha/Algorithmic-NIFTY-500-Screener-Execution-Engine/blob/main/NIFTY500_Screener.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas
!pip install mplfinance pandas
!pip install dhanhq
!pip install pandas_ta

In [ ]:
from dhanhq import dhanhq, DhanContext

# 🔐 API CREDENTIALS
CLIENT_ID = ""
ACCESS_TOKEN = ""

# Initialize Dhan SDK
print("🔄 Authenticating with Dhan...")
dhan_context = DhanContext(CLIENT_ID, ACCESS_TOKEN)
dhan = dhanhq(dhan_context)
print("✅ Authentication Successful!")

In [ ]:
import pandas as pd
import pandas_ta as ta
import numpy as np
import time
import requests
import io
from datetime import datetime, timedelta

# ==========================================
# 1. Fetch & Filter Data (Using User's Local File)
# ==========================================
def get_nifty500_dhan_mapping():
    print("[INFO] Fetching Official NIFTY 500 Constituents from NSE...")

    # 1A. Fetch official NIFTY 500 list from NSE
    nse_url = "https://www.niftyindices.com/IndexConstituent/ind_nifty500list.csv"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

    try:
        response = requests.get(nse_url, headers=headers, timeout=10)
        nifty500_df = pd.read_csv(io.StringIO(response.text))
        nifty_symbols = nifty500_df['Symbol'].tolist()
    except Exception as e:
        print(f"[ERROR] Failed to fetch NIFTY 500 list: {e}")
        return pd.DataFrame()

    print("[INFO] Reading your local NSE Symbol List...")
    # 1B. Read your local Excel file (Updated for .xlsx)
    local_file_path = "/content/Symbol List NSE.xlsx"

    try:
        # Use read_excel instead of read_csv
        master_df = pd.read_excel(local_file_path)
    except Exception as e:
        print(f"[ERROR] Could not read local file '{local_file_path}': {e}")
        # Fallback check in case the file is actually a CSV that was just named poorly
        try:
            print("[INFO] Attempting to read as a CSV with latin1 encoding instead...")
            master_df = pd.read_csv(local_file_path, encoding='latin1')
        except Exception as fallback_e:
            print(f"[ERROR] Fallback failed: {fallback_e}")
            return pd.DataFrame()

    # 1C. Rename your columns to match the standard API variables
    master_df = master_df.rename(columns={
        'Trading Symbol': 'SEM_CUSTOM_SYMBOL',
        'Secuirty ID': 'SEM_SMST_SECURITY_ID'  # Matching your exact spelling
    })

    # 1D. Strictly match your local file's symbols with the official NIFTY 500 list
    mapped_df = master_df[master_df['SEM_CUSTOM_SYMBOL'].isin(nifty_symbols)]

    print(f"[INFO] Successfully mapped {len(mapped_df)} NIFTY 500 stocks from your file.")
    return mapped_df

# ==========================================
# 2. Core Strategy Logic (HVN + OBV + Strict Filters)
# ==========================================
def analyze_stock(df, symbol):
    # Need at least 200 days of data for the 200 SMA
    if df.empty or len(df) < 200:
        return None

    current_price = df['close'].iloc[-1]
    current_high = df['high'].iloc[-1]
    current_low = df['low'].iloc[-1]

    # -- FILTER 1: Liquidity & Price Baseline --
    avg_vol_30 = df['volume'].tail(30).mean()
    if current_price < 50 or avg_vol_30 < 500000:
        return None

    # -- FILTER 2: Structural Trend --
    df['SMA_200'] = ta.sma(df['close'], length=200)
    sma_200 = df['SMA_200'].iloc[-1]

    if current_price < sma_200:
        return None # Stock is in a long-term downtrend; skip it.

    # -- FILTER 3: Actionable Bounce (Candlestick Rejection) --
    daily_range = current_high - current_low
    if daily_range > 0:
        close_strength = (current_price - current_low) / daily_range
        if close_strength < 0.70:
            return None # Buyers didn't defend the level strongly enough today

    # -- FILTER 4: HVN Demand Zone --
    volume_profile, price_bins = np.histogram(
        df['close'],
        bins=50,
        weights=df['volume']
    )

    top_hvn_indices = np.argsort(volume_profile)[-3:]
    in_zone = False
    target_hvn = 0

    for idx in top_hvn_indices:
        hvn_price = price_bins[idx]
        if abs(current_price - hvn_price) / hvn_price <= 0.02:
            in_zone = True
            target_hvn = hvn_price
            break

    if not in_zone:
        return None

    # -- FILTER 5: OBV Momentum --
    df['OBV'] = ta.obv(close=df['close'], volume=df['volume'])
    obv_20d = df['OBV'].tail(20)
    obv_trending_up = obv_20d.iloc[-1] > obv_20d.iloc[0]

    if in_zone and obv_trending_up:
        return {
            'Symbol': symbol,
            'Current Price': round(current_price, 2),
            'HVN Support': round(target_hvn, 2),
            '200 SMA': round(sma_200, 2),
            '30d Avg Vol': int(avg_vol_30),
            'Status': 'Confirmed Setup'
        }
    return None

# ==========================================
# 3. Main Screener Execution Loop
# ==========================================
def run_screener():
    # ASSUMPTION: The 'dhan' object is already authorized and available globally in your Jupyter/Python environment
    global dhan

    mapped_df = get_nifty500_dhan_mapping()

    if mapped_df.empty:
        print("[ERROR] Mapping failed. Exiting.")
        return

    results = []

    # Setup the 3-Year Date Range
    end_date = datetime.now()
    start_date = end_date - timedelta(days=3*365)
    str_from_date = start_date.strftime('%Y-%m-%d')
    str_to_date = end_date.strftime('%Y-%m-%d')

    print(f"\n[INFO] Starting Scan of NIFTY 500 stocks...")

    for index, row in mapped_df.iterrows():
        symbol = row['SEM_CUSTOM_SYMBOL']
        sec_id = str(row['SEM_SMST_SECURITY_ID'])

        try:
            response = dhan.historical_daily_data(
                security_id=sec_id,
                exchange_segment='NSE_EQ',
                instrument_type='EQUITY',
                expiry_code=0,
                from_date=str_from_date,
                to_date=str_to_date
            )

            if response['status'] == 'success' and 'data' in response and response['data']:
                df = pd.DataFrame(response['data'])

                result = analyze_stock(df, symbol)
                if result:
                    results.append(result)
                    print(f"[🔥 ALERT] Setup Found: {symbol} at ₹{result['Current Price']}")

            # API Rate Limit Safety: 4 requests per second
            time.sleep(0.25)

        except Exception as e:
            pass

    # Output Results
    if results:
        final_df = pd.DataFrame(results)
        final_df.to_csv('NIFTY500_Strict_Setups.csv', index=False)
        print(f"\n[SUCCESS] Scan complete. Found {len(results)} high-probability setups. Saved to 'NIFTY500_Strict_Setups.csv'.")
    else:
        print("\n[INFO] Scan complete. Market is likely weak; no stocks met all strict criteria today.")

if __name__ == "__main__":
    run_screener()

In [ ]:
import pandas as pd
import pandas_ta as ta
import numpy as np
import time
from datetime import datetime, timedelta

def get_target_stocks_from_csv():
    print("[INFO] Loading high-probability setups from Daily Screener output...")
    try:
        # Read the file generated by our Daily Screener
        setups_df = pd.read_csv('/content/NIFTY500_Strict_Setups.csv')
        target_symbols = setups_df['Symbol'].tolist()
        print(f"[INFO] Found {len(target_symbols)} stocks ready for intraday analysis.")

        # Robust Master List Loading
        local_file_path = "/content/Symbol List NSE.xlsx"
        csv_fallback_path = "Symbol List NSE.xlsx - Sheet1.csv"

        try:
            master_df = pd.read_excel(local_file_path)
        except Exception:
            try:
                master_df = pd.read_csv(csv_fallback_path, encoding='latin1')
            except Exception as e:
                print(f"[ERROR] Could not load Master Symbol List: {e}")
                return pd.DataFrame()

        master_df = master_df.rename(columns={
            'Trading Symbol': 'SEM_CUSTOM_SYMBOL',
            'Secuirty ID': 'SEM_SMST_SECURITY_ID'
        })

        target_df = master_df[master_df['SEM_CUSTOM_SYMBOL'].isin(target_symbols)]
        return target_df

    except FileNotFoundError:
        print("[ERROR] 'NIFTY500_Strict_Setups.csv' not found. Run the daily screener first.")
        return pd.DataFrame()

def generate_trade_parameters(df, symbol):
    if df.empty or len(df) < 50:
        return None

    current_price = df['close'].iloc[-1]

    # 1. Calculate Volatility (ATR) for Stoploss
    df['ATR'] = ta.atr(df['high'], df['low'], df['close'], length=14)
    current_atr = df['ATR'].iloc[-1]

    # 25-min timeframe adjustment: 15 candles = 1 full NSE trading day
    daily_atr = current_atr * np.sqrt(15)

    stop_loss = current_price - (1.5 * current_atr)
    risk_per_share = current_price - stop_loss

    # 2. Define Target (1:2.5 R:R)
    target_price = current_price + (risk_per_share * 2.5)
    target_distance = target_price - current_price
    estimated_days = target_distance / daily_atr if daily_atr > 0 else 0

    # 3. Calculate Intraday Momentum Indicators
    macd = ta.macd(df['close'], fast=12, slow=26, signal=9)
    df = pd.concat([df, macd], axis=1)

    macd_line = df['MACD_12_26_9'].iloc[-1]
    macd_signal = df['MACDs_12_26_9'].iloc[-1]
    macd_hist_current = df['MACDh_12_26_9'].iloc[-1]
    macd_hist_prev = df['MACDh_12_26_9'].iloc[-2]

    df['RSI'] = ta.rsi(df['close'], length=14)
    current_rsi = df['RSI'].iloc[-1]

    # VWAP Approximation
    df.set_index(pd.DatetimeIndex(df.index), inplace=True)
    df['VWAP'] = ta.vwap(high=df['high'], low=df['low'], close=df['close'], volume=df['volume'])
    current_vwap = df['VWAP'].iloc[-1]

    # 4. PROBABILITY / CONFIDENCE SCORING ALGORITHM
    probability_score = 50 # Base score (Survived Daily filters)

    if current_price > current_vwap: probability_score += 15
    if macd_line > macd_signal: probability_score += 15
    if macd_hist_current > macd_hist_prev and macd_hist_current > 0: probability_score += 10
    if 50 <= current_rsi <= 70: probability_score += 10

    action = "ENTER NOW" if probability_score >= 80 else "WAIT / MONITOR"

    return {
        'Symbol': symbol,
        'Entry': round(current_price, 2),
        'Target': round(target_price, 2),
        'Stoploss': round(stop_loss, 2),
        'Hold Days': f"{int(round(estimated_days))}d",
        'Confidence': f"{probability_score}%",
        'Action': action
    }

def run_execution_engine():
    # ASSUMPTION: dhan object is globally available
    global dhan

    target_df = get_target_stocks_from_csv()
    if target_df.empty: return

    results = []

    # 5 days provides exactly 75 candles for the 25-minute chart
    end_date = datetime.now()
    start_date = end_date - timedelta(days=90)
    str_from_date = start_date.strftime('%Y-%m-%d')
    str_to_date = end_date.strftime('%Y-%m-%d')

    print("\n[INFO] Running Deep Dive Execution Engine (25-Min Algorithmic Timeframe)...\n")
    print(f"{'SYMBOL':<15} | {'ENTRY':<8} | {'TARGET':<8} | {'STOP':<8} | {'HOLD':<6} | {'WIN PROB':<8} | {'ACTION'}")
    print("-" * 85)

    for index, row in target_df.iterrows():
        symbol = row['SEM_CUSTOM_SYMBOL']
        sec_id = str(row['SEM_SMST_SECURITY_ID'])

        try:
            response = dhan.intraday_minute_data(
                security_id=sec_id,
                exchange_segment='NSE_EQ',
                instrument_type='EQUITY',
                from_date=str_from_date,
                to_date=str_to_date,
                interval='25'
            )

            if response['status'] == 'success' and 'data' in response and response['data']:
                df = pd.DataFrame(response['data'])
                print(f"   -> [DEBUG] {symbol} returned {len(df)} candles.")
                trade_setup = generate_trade_parameters(df, symbol)

                if trade_setup:
                    results.append(trade_setup)
                    star = "⭐" if trade_setup['Confidence'] in ['80%', '90%', '100%'] else "  "
                    print(f"{trade_setup['Symbol']:<15} | {trade_setup['Entry']:<8} | {trade_setup['Target']:<8} | {trade_setup['Stoploss']:<8} | {trade_setup['Hold Days']:<6} | {trade_setup['Confidence']:<8} | {star}{trade_setup['Action']}")

            time.sleep(0.5)

        except Exception as e:
            print(f"   -> [DEBUG] Skipped {symbol} due to error: {e}")

    if results:
        final_df = pd.DataFrame(results)
        final_df.to_csv('Final_Trade_Executions.csv', index=False)
        print("\n[SUCCESS] Final execution plan saved to 'Final_Trade_Executions.csv'.")

if __name__ == "__main__":
    run_execution_engine()